# GSE139088 clustering

In [ ]:
import time
from pathlib import Path
import pandas as pd
import scanpy as sc
import anndata as ad
from matplotlib import pyplot as plt
import re
import sys
import session_info
import os

In [ ]:
plt.rcParams['figure.figsize'] = (3,3)
#plt.rcParams['figure.dpi'] = 500

In [ ]:
print('active conda environment: ', os.path.basename(sys.prefix))

In [ ]:
# Directories
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR
print('BASE_DIR:', BASE_DIR)

# input ref data
input_dir = BASE_DIR / "data" / "h5ad" / "03_scvi"

h5ad_out_dir = BASE_DIR / "data" / "h5ad" / "04_clustered"
h5ad_out_dir.mkdir(parents=True, exist_ok=True)

# output file
h5ad_out = h5ad_out_dir / "GSE139088-scvi-leiden.h5ad"

In [ ]:
###

In [ ]:
adata = sc.read_h5ad(input_dir / "GSE139088-scvi.h5ad")

In [ ]:
adata.obs

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI', random_state = 0) # Use latent representation to build neighbors graph
sc.tl.umap(adata, random_state = 0)

In [ ]:
sc.tl.leiden(adata, key_added='leiden', resolution=1)

In [ ]:
sc.pl.umap(adata, color = ['sample_id'], legend_fontsize = 10)
sc.pl.umap(adata, color = ['leiden'], legend_fontsize = 10)

# DE

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
df = sc.get.rank_genes_groups_df(adata, group = None)
df = df.copy()
df = df[(df.pvals_adj < 0.05) & (df.logfoldchanges > .5)].copy()
df.head(5)

sc.pl.rank_genes_groups(adata, groupby = 'leiden', method = 'wilcoxon')

In [ ]:
sc.pl.umap(adata, color = ['total_counts', 'n_genes_by_counts', 'sample_id', 'pct_counts_mt', 'pct_counts_ribosomal'], legend_fontsize = 10)

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color = ['Rbfox3', 'Th', 'Piezo1', 'Piezo2', 'Aif1', 'P2ry12', 'Itgam', 'Trpv1', 'original_annotation', 'pct_counts_mt', 'pct_counts_ribosomal', 'doublet_scores'], legend_fontsize = 10)

In [ ]:
sc.pl.umap(adata, color = ['original_annotation'], title = "Author Annotations", frameon = False, size = 4)

In [ ]:
sc.pl.umap(adata, color = ['leiden'], title = "Cluster", frameon = False, size = 4)

In [ ]:
sc.pl.umap(adata, color = ['Rbfox3', 'Mbp', 'Th'], frameon = False, size = 4)

In [ ]:
sc.pl.umap(adata, color = ['Trpv1'], frameon = False, size = 4)

In [ ]:
# Number of cells
n_cells = adata.n_obs

# Median total counts per cell (transcript count)
median_total_counts = adata.obs['total_counts'].median()

# Median number of genes detected per cell
median_n_genes = adata.obs['n_genes_by_counts'].median()

# Print results
print(f"Number of cells: {n_cells:,}")
print(f"Median total counts per cell: {median_total_counts:.0f}")
print(f"Median number of detected genes per cell: {median_n_genes:.0f}")

## Export

In [ ]:
h5ad_out

In [ ]:
adata.write_h5ad(h5ad_out, compression='gzip')